In [1]:
from dlfs.model import TransformerDecoderModel

from dlfs.loss import CCE_Loss
from dlfs.optimizers import Optimizer_Adam

import numpy as np

# Tiny Shakespeare dataset

In [2]:
with open('../../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Number of characters in the whole text: {len(text)}')

Number of characters in the whole text: 1115394


In [3]:
print(f'{text[:200]}')

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [4]:
chars = sorted(list(set(text))) # get sorted unique characters
vocab_size = len(chars) # number of unique characters
print(f'All characters: {"".join(chars)}')
print(f'Vocab size: {vocab_size}')

All characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65


# Character level tokenizer

In [5]:
encoded_dict = {chars[i]: i for i in range(vocab_size)} # map character to int
decoded_dict = {i: chars[i] for i in range(vocab_size)} # map int to character

def encode(s: str) -> list[int]:
    return [encoded_dict[c] for c in s]

def decode(s: list[int]) -> str:
    return "".join([decoded_dict[c] for c in s])

print(encode("test string"))
print(decode(encode("test string")))

[58, 43, 57, 58, 1, 57, 58, 56, 47, 52, 45]
test string


# Convert whole dataset to indices

In [6]:
data = encode(text)
data = np.array(data, dtype=np.int32)
print(data.shape)

(1115394,)


# Train test split

In [7]:
n = int(0.9*len(data))
X_train, X_test = data[:n], data[n:]
print(f'n: {n}\nX_train: {X_train.shape}\nX_test: {X_test.shape}')

n: 1003854
X_train: (1003854,)
X_test: (111540,)


# Creating sequences

In [8]:
def create_sequences(data, seq_len = 8):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+1:i+seq_len+1])
    return np.array(X), np.array(y)

seq_len = 64

X_train, y_train = create_sequences(X_train, seq_len)
print(f'{X_train.shape}, {y_train.shape}')
print(f'There are {X_train.shape[0]} sequences, each sequence is of length {X_train.shape[1]}')

(1003790, 64), (1003790, 64)
There are 1003790 sequences, each sequence is of length 64


# Example input sequence and target sequence

In [9]:
print(X_train[0, :5]) # first couple of tokens
print(y_train[0, :5])

[18 47 56 57 58]
[47 56 57 58  1]


# Model training

In [10]:
np.random.seed(1337)

batch_size = 64

n_embed = 192
n_head = 6
n_dec_layers = 3
dim_ff = 768
dropout = 0.2
eps = 1e-5

epochs = 50
lr = 3e-4

loss = CCE_Loss(from_logits=True)
optimizer = Optimizer_Adam(learning_rate=lr, decay=0., clip_grad=False)

model = TransformerDecoderModel(vocab_size=vocab_size, 
                                seq_len=seq_len, 
                                n_embed=n_embed, 
                                n_head=n_head,
                                n_dec_layers=n_dec_layers,
                                dim_ff=dim_ff, 
                                dropout=dropout,
                                layer_norm_eps=eps,
                                loss_function=loss, 
                                optimizer=optimizer)

model.train(X_train, y_train, print_every=5, epochs=epochs, batch_size=batch_size)

===== EPOCH : 0 ===== LOSS : 4.362040432438005 =====
===== EPOCH : 5 ===== LOSS : 3.6394020993213863 =====
===== EPOCH : 10 ===== LOSS : 3.486691712379854 =====
===== EPOCH : 15 ===== LOSS : 3.4254558945820848 =====
===== EPOCH : 20 ===== LOSS : 3.3681236117272197 =====
===== EPOCH : 25 ===== LOSS : 3.402481893880401 =====
===== EPOCH : 30 ===== LOSS : 3.329123471881199 =====
===== EPOCH : 35 ===== LOSS : 3.3708356247285702 =====
===== EPOCH : 40 ===== LOSS : 3.3245194993286376 =====
===== EPOCH : 45 ===== LOSS : 3.374141312366778 =====
===== EPOCH : 50 ===== LOSS : 3.3230178448720276 =====


In [11]:
print(decode(model.generate(np.zeros((1, 1), dtype=np.int32), seq_len, max_new_tokens=500)[0].tolist()))


gtdattglglad
rsiia ..aelsjmoealn yrdfbons O
!reeen s
  hcn' llt mheufeoc  doew   Gtdate eeJ, Kq oofeliaodcd
 onAh  o e auo ittatRoaar i 
e  h a e;yt
 t;y  ,nreiw ar Woafsoeyrgsh?reepot reen.au lerlwt s lse
WOo rg tbir idelXt clotretdi  aistebdaihm en ,Hue es,dtohultss Cr, dwogt t bmo 
 khmecdhhkos ep uel r  ,
ret:notase  hsiafo dfa pbA hrcTsaighqsih:di.a de itnt  B eeehh :Avr  o
mieOni mohUen nuas  ias tOhshmajUOl Nhg sbte leo  systbittoec
IsO y   honaairdoa aphsswhd  dsJL   dbhlhdw
wtatoaIgnp a
